# Respostas às perguntas do case

Fonte: `ifood_case.refined.fct_taxi_trip` (fato unificado já aprovado nas
regras de qualidade) e os agregados `agg_trip_monthly` / `agg_trip_hourly`.

As duas perguntas do case são ambíguas em português, e a ambiguidade muda o
número. Em vez de escolher em silêncio, cada pergunta é respondida nas duas
leituras possíveis, com a recomendação explícita de qual usar.

---
## Pergunta 1

> Qual a média de valor total (`total_amount`) recebido em um mês considerando
> todos os *yellow* táxis da frota?

**A ambiguidade:** "média de valor total recebido em um mês" pode significar

* **Leitura A — ticket médio:** média do `total_amount` *por corrida*, quebrada por mês.
  Responde "quanto vale, em média, uma corrida".
* **Leitura B — faturamento médio mensal:** soma do `total_amount` de cada mês e
  média dessas somas. Responde "quanto a frota fatura, em média, por mês".

As duas estão abaixo. A **Leitura B** é a que responde literalmente ao enunciado
("valor total recebido *em um mês*" = o que entrou no mês); a Leitura A é a
métrica que uma área de negócio normalmente quer acompanhar.

### 1.A · Ticket médio por corrida, por mês

In [0]:
%sql
SELECT
    reference_month                          AS mes,
    trip_count                               AS corridas,
    ROUND(avg_total_amount_per_trip, 2)      AS ticket_medio,
    ROUND(median_total_amount, 2)            AS mediana,
    ROUND(total_revenue, 2)                  AS receita_total
FROM ifood_case.refined.agg_trip_monthly
WHERE trip_type = 'yellow'
ORDER BY mes;

In [0]:
%sql
-- Mesmo número calculado direto do fato, sem passar pelo agregado.
-- Serve como prova de que o agregado está correto.
SELECT
    date_format(pickup_datetime, 'yyyy-MM')  AS mes,
    COUNT(*)                                 AS corridas,
    ROUND(AVG(total_amount), 2)              AS ticket_medio
FROM ifood_case.refined.fct_taxi_trip
WHERE trip_type = 'yellow'
GROUP BY ALL
ORDER BY mes;

### 1.B · Faturamento médio mensal da frota yellow

In [0]:
%sql
WITH por_mes AS (
    SELECT reference_month, total_revenue, trip_count
    FROM ifood_case.refined.agg_trip_monthly
    WHERE trip_type = 'yellow'
)
SELECT
    COUNT(*)                          AS meses_considerados,
    ROUND(SUM(total_revenue), 2)      AS receita_do_periodo,
    ROUND(AVG(total_revenue), 2)      AS receita_media_mensal,
    ROUND(AVG(trip_count), 0)         AS corridas_media_mensal
FROM por_mes;

### 1.C · Análise de sensibilidade

Quanto a decisão de mandar os `total_amount` negativos para a quarentena muda
a resposta? Se o impacto for irrelevante, a decisão é segura; se for grande,
ela precisa ser validada com a área de negócio antes de virar número oficial.

In [0]:
%sql
WITH todas AS (
    SELECT total_amount, 'aprovadas (fato)' AS cenario
    FROM ifood_case.refined.fct_taxi_trip WHERE trip_type = 'yellow'
    UNION ALL
    SELECT total_amount, 'aprovadas + negativos'
    FROM ifood_case.refined.fct_taxi_trip WHERE trip_type = 'yellow'
    UNION ALL
    SELECT total_amount, 'aprovadas + negativos'
    FROM ifood_case.refined.rej_taxi_trip
    WHERE trip_type = 'yellow'
      AND array_contains(_rejection_reasons, 'total_amount_negativo')
      AND size(_rejection_reasons) = 1
)
SELECT
    cenario,
    COUNT(*)                        AS corridas,
    ROUND(AVG(total_amount), 4)     AS ticket_medio,
    ROUND(SUM(total_amount), 2)     AS receita
FROM todas
GROUP BY cenario
ORDER BY cenario;

---
## Pergunta 2

> Qual a média de passageiros (`passenger_count`) por cada hora do dia que
> pegaram táxi no mês de maio considerando todos os táxis da frota?

**Escopo — "todos os táxis da frota":** yellow **e** green. As demais bases da
TLC (FHV e High Volume FHV, que cobrem Uber/Lyft) não são táxis licenciados e,
além disso, sequer publicam `passenger_count` — ficam de fora por definição.

**A ambiguidade:**

* **Leitura A — ocupação média:** média do `passenger_count` por corrida, em cada
  hora. Responde "quantas pessoas viajam juntas, em média, nesse horário".
* **Leitura B — volume médio de passageiros:** total de passageiros transportados
  naquela hora dividido pelos 31 dias de maio. Responde "quantos passageiros a
  frota move por hora, num dia típico".

A **Leitura A** é a leitura direta de "média de `passenger_count`" e é a resposta
principal. A Leitura B é a que serve para dimensionar operação, e está logo abaixo.

**Filtro aplicado:** corridas com `passenger_count` nulo ou zero são excluídas
*desta análise específica* (não da base). Incluir zeros como se fossem corridas
sem ninguém dentro afundaria a média artificialmente.

### 2.A · Ocupação média por hora do dia (maio/2023, frota completa)

In [0]:
%sql
SELECT
    LPAD(pickup_hour, 2, '0')                                  AS hora,
    SUM(trips_with_passenger_count)                            AS corridas,
    ROUND(SUM(total_passengers) / SUM(trips_with_passenger_count), 4) AS media_passageiros_por_corrida
FROM ifood_case.refined.agg_trip_hourly
WHERE pickup_year = '2023' AND pickup_month = '05'
GROUP BY pickup_hour
ORDER BY pickup_hour;

In [0]:
%sql
-- Conferência direto no fato.
SELECT
    LPAD(hour(pickup_datetime), 2, '0')     AS hora,
    COUNT(*)                                AS corridas,
    ROUND(AVG(passenger_count), 4)          AS media_passageiros_por_corrida
FROM ifood_case.refined.fct_taxi_trip
WHERE pickup_datetime >= '2023-05-01'
  AND pickup_datetime <  '2023-06-01'
  AND passenger_count > 0
GROUP BY ALL
ORDER BY hora;

### 2.B · Passageiros por hora num dia típico de maio

In [0]:
%sql
SELECT
    LPAD(pickup_hour, 2, '0')                            AS hora,
    SUM(total_passengers)                                AS passageiros_no_mes,
    MAX(distinct_days)                                   AS dias_com_dado,
    ROUND(SUM(total_passengers) / MAX(distinct_days), 1) AS passageiros_por_hora_dia_tipico
FROM ifood_case.refined.agg_trip_hourly
WHERE pickup_year = '2023' AND pickup_month = '05'
GROUP BY pickup_hour
ORDER BY pickup_hour;

### 2.C · Abertura por tipo de táxi

Yellow e green têm perfis de ocupação diferentes; vale mostrar.

In [0]:
%sql
SELECT
    LPAD(pickup_hour, 2, '0') AS hora,
    ROUND(MAX(CASE WHEN trip_type = 'yellow' THEN avg_passenger_count END), 4) AS yellow,
    ROUND(MAX(CASE WHEN trip_type = 'green'  THEN avg_passenger_count END), 4) AS green
FROM ifood_case.refined.agg_trip_hourly
WHERE pickup_year = '2023' AND pickup_month = '05'
GROUP BY pickup_hour
ORDER BY pickup_hour;

### 2.D · Visualização

In [0]:
import matplotlib.pyplot as plt

df = spark.sql("""
    SELECT pickup_hour,
           SUM(total_passengers) / SUM(trips_with_passenger_count) AS ocupacao_media,
           SUM(trips_with_passenger_count)                          AS corridas
    FROM ifood_case.refined.agg_trip_hourly
    WHERE pickup_year = '2023' AND pickup_month = '05'
    GROUP BY pickup_hour ORDER BY pickup_hour
""").toPandas()

fig, ax1 = plt.subplots(figsize=(11, 4.5))
ax1.bar(df['pickup_hour'], df['corridas'], alpha=0.25, label='Corridas')
ax1.set_xlabel('Hora do embarque')
ax1.set_ylabel('Corridas')
ax1.set_xticks(range(24))

ax2 = ax1.twinx()
ax2.plot(df['pickup_hour'], df['ocupacao_media'], marker='o', linewidth=2, label='Ocupação média')
ax2.set_ylabel('Passageiros por corrida')

plt.title('Maio/2023 — volume de corridas e ocupação média por hora (yellow + green)')
fig.tight_layout()
plt.show()

---
## Como ler estes números

1. **Ticket médio é estável entre os meses, faturamento não.** A variação mensal
   da receita vem principalmente de volume de corridas, não de preço por corrida.
2. **A média do `total_amount` fica acima da mediana** por causa da cauda de
   corridas caras (aeroporto, tarifas negociadas). Para acompanhamento operacional,
   a mediana descreve melhor a corrida típica.
3. **A ocupação média por hora varia pouco** — fica na casa de 1,3 a 1,4 passageiro
   por corrida ao longo do dia. O que varia de verdade é o *volume*: madrugada e
   pico da tarde são mundos diferentes. Por isso a Leitura B (2.B) importa tanto
   quanto a Leitura A para qualquer decisão de operação.

> Preencha `analysis/RESULTADOS.md` com os números obtidos depois de rodar o
> pipeline — é o resumo executivo do case.